### Script to combine the systems of UKMO into one anomaly timeseries.  Assumes anomalies from lead dependent climatology for each system have already been calculated

In [1]:
import xarray as xr
import numpy as np
import pandas
import yaml
import os
import xesmf as xe

In [2]:
model='UKMO'
with open("../../../DATA_DOWNLOAD/system_year_info.yaml") as f:
    model_info = yaml.safe_load(f)
info = model_info[model]
systems = info['forecast'].keys()

#### Information e.g., variable to use, start and end years, start and end years of hindcast period

In [3]:
var = 'slp'
yhind1 = 1993 ; yhind2 = 2016
basepath="/glade/campaign/cgd/cas/islas/DATASETS/CDS/seasonal/"+model+"/nc/"+var+"/"
outpath="/glade/campaign/cgd/cas/islas/DATASETS/CDS/seasonal/combine_systems/"+model+"/"

# Outputting also to the same location where the other data are
outpath2="/glade/campaign/cgd/cas/islas/python_savs/rossbypalooza26/remove_lead_dependent_climo/"+model+"/"
os.makedirs(outpath, exist_ok=True)

#### Select the last system as the hindcast system

#### Read in the hindcast data and the forecast data.  Combined and check all years are there
#### Also regridding to a common 1 degree grid because the systems are not necessarily all on the same grid

In [5]:
grid_out = xr.Dataset({'lat':(['lat'], np.arange(-90,90,1))}, {'lon': (['lon'], np.arange(0,360,1))})

In [ ]:
### Loop over systems 
for isys in systems:

    sysbase=isys
    alldat_ALLM=[]
    
    print(isys)
    dat = xr.open_dataset(basepath+str(isys)+'/hindcast_anoms_ALLM_'+var+'.nc')
    regridder = xe.Regridder(dat, grid_out, 'bilinear', periodic=True, reuse_weights=False,
                             filename='/glade/derecho/scratch/islas/temp/wgt.nc')
    dat_rg = regridder(dat)
    alldat_ALLM.append(dat_rg)

    for isys in systems:    
        print(isys)
        years = info['forecast'][isys]
        dat = xr.open_dataset(basepath+str(isys)+'/forecast_anoms_ALLM_'+var+'.nc')
        regridder = xe.Regridder(dat, grid_out, 'bilinear', periodic=True, reuse_weights=False,
                             filename='/glade/derecho/scratch/islas/temp/wgt.nc')
        dat_rg = regridder(dat)
        alldat_ALLM.append(dat_rg.sel(year=years))

    alldat_ALLM = xr.concat(alldat_ALLM, dim='year')
    alldat_ALLM = alldat_ALLM.transpose("member","year","lead","lat","lon")

    alldat_ALLM.to_netcdf(outpath2+var+'_anoms_ALLM_'+
            str(int(np.min(alldat_ALLM.year)))+'_'+str(int(np.max(alldat_ALLM.year)))+'_'+str(sysbase)+'.nc')